# Model Comparison and Feature Importance Analysis

This notebook reproduces the machine learning analyses presented in the accompanying study.

The notebook was developed and executed in the Kaggle Notebook environment. Input paths assume the corresponding datasets are available as Kaggle datasets.

# Imports

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")
import re
import optuna
from optuna.samplers import TPESampler
import pickle
import matplotlib.pyplot as plt
import scipy.stats as stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import shap

# Data Preparation

In [ ]:
np.set_printoptions(threshold=np.inf)

# Load dataset
features_matrix = pd.read_csv('../input/wt25-dataset/Xtbl_wt25.csv')
X = features_matrix.drop(columns=['transcript', 'random_factor', 'random_num'])

df = pd.read_csv('../input/wt25-dataset/Ytbl_wt25.csv')
y = df['RE'].copy()

categorical_cols = [col for col in X.columns if X[col].dtype == 'object' or X[col].dtype.name == 'category']

# Setup OneHotEncoder
encoder = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', encoder, categorical_cols),
    ],
    remainder='passthrough'  # Include this to keep numerical columns unchanged
)

# Fit the preprocessor on the entire dataset
preprocessor.fit(X)

# Tuning

In [ ]:
# XGB

def return_rmse(dtrain, dtest, params):
    model = xgb.train(params, dtrain, num_boost_round=600, evals=[(dtest, 'eval')], early_stopping_rounds=100,
                  verbose_eval=0)
    result = model.eval(dtest)
    result = float(re.search(r'[\d.]+$', result).group(0))
    return result

def objective(dtrain, dtest, trial):
    params = {'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 10.0),
    'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 10.0),
    'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
    'gamma': trial.suggest_float('gamma', 0, 5),
    'subsample': trial.suggest_float('subsample', 0.5, 1.0),
    'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
    'max_depth': trial.suggest_int('max_depth', 3, 10),
    'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    'min_split_loss': trial.suggest_float('min_split_loss', 0, 5),
    'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
             'nthread': -1}
    return return_rmse(dtrain, dtest, params)

def hyperparam_tuning(X_train, y_train, X_test, y_test):
    res=[]
    dtrain = xgb.DMatrix(X_train, y_train, enable_categorical=True)
    dtest = xgb.DMatrix(X_test, y_test, enable_categorical=True)
    res.append(return_rmse(dtrain, dtest, params={}))

    xgb_study = optuna.create_study(direction='minimize',sampler =TPESampler() )
    xgb_study.optimize(lambda trial: objective(dtrain, dtest, trial), n_trials=100, show_progress_bar=True)
    
    return xgb_study


# lr

def ridge_objective(X_train, y_train, X_test, y_test, trial):
    alpha = trial.suggest_float("alpha", 0.01, 100.0, log=True)

    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    return rmse

def tune_ridge_with_optuna(X_train, y_train, X_test, y_test):
    lr_study = optuna.create_study(direction="minimize", sampler=TPESampler())
    lr_study.optimize(lambda trial: ridge_objective(X_train, y_train, X_test, y_test, trial), 
                   n_trials=50, 
                   show_progress_bar=True)
    
    return lr_study


# lasso

def lasso_objective(X_train, y_train, X_test, y_test, trial):
    alpha = trial.suggest_float("alpha", 0.0001, 1.0, log=True)

    model = Lasso(alpha=alpha, random_state=42, max_iter=10000)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    return rmse

def tune_lasso_with_optuna(X_train, y_train, X_test, y_test):
    lasso_study = optuna.create_study(direction="minimize", sampler=TPESampler())
    lasso_study.optimize(lambda trial: lasso_objective(X_train, y_train, X_test, y_test, trial), 
                   n_trials=50, 
                   show_progress_bar=True)
    
    return lasso_study


# rf

def rf_objective(X_train, y_train, X_test, y_test, trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'random_state': 42,
        'n_jobs': -1
    }

    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    return rmse

def tune_rf_with_optuna(X_train, y_train, X_test, y_test):
    rf_study = optuna.create_study(direction="minimize", sampler=TPESampler())
    rf_study.optimize(lambda trial: rf_objective(X_train, y_train, X_test, y_test, trial), 
                   n_trials=50, 
                   show_progress_bar=True)
    
    return rf_study

# Train Models

In [ ]:
# Function to calculate metrics

def train_models(X_train, y_train, X_test, y_test, xgb_study):

    lr_study = tune_ridge_with_optuna(X_train, y_train, X_test, y_test)
    best_alpha = lr_study.best_params['alpha']
    lr_pipeline = Pipeline([
        ('regressor', Ridge(alpha=best_alpha))
    ])
    lr_pipeline.fit(X_train, y_train)


    # Define the Lasso regression pipeline
    lasso_study = tune_lasso_with_optuna(X_train, y_train, X_test, y_test)
    best_lasso_alpha = lasso_study.best_params['alpha']
    
    lasso_pipeline = Pipeline([
        ('regressor', Lasso(alpha=best_lasso_alpha, random_state=42, max_iter=10000))
    ])
    lasso_pipeline.fit(X_train, y_train)

    
    # Random Forest
    rf_study = tune_rf_with_optuna(X_train, y_train, X_test, y_test)
    rf_params = rf_study.best_params
    
    rf_pipeline = Pipeline([
        ('regressor', RandomForestRegressor(**rf_params))
    ])
    rf_pipeline.fit(X_train, y_train)

    
    # XGBoost
    xgb_model = xgb.XGBRegressor(objective='reg:squarederror', **xgb_study.best_params)  # Assuming best_params are already set
    xgb_model.fit(X_train, y_train)
    
    return {
        'lr': {'model': lr_pipeline},
        'lasso': {'model': lasso_pipeline},
        'rf': {'model': rf_pipeline},
        'xgb': {'model': xgb_model}
    }

# Evaluate the Models

In [ ]:
def evaluate_model(model, X, y):
    predictions = model.predict(X)
    rmse = mean_squared_error(y, predictions, squared=False)
    mae = mean_absolute_error(y, predictions)
    r2 = r2_score(y, predictions)
    spearman_corr, _ = stats.spearmanr(y, predictions)
    pearson_corr, _ = stats.pearsonr(y, predictions)
    return rmse, mae, r2, spearman_corr, pearson_corr, y, predictions


def evaluate_models(model_dict, X_test, y_test):
    results = {}
    for model in model_dict:
        rmse, mae, r2, spearman_corr, pearson_corr, y, pred = (
            evaluate_model(model_dict[model]['model'], X_test, y_test)
        )
        
        model_dict[model]['rmse'] = rmse        
        model_dict[model]['mae'] = mae        
        model_dict[model]['r2'] = r2
        model_dict[model]['spearman_corr'] = spearman_corr        
        model_dict[model]['pearson_corr'] = pearson_corr

        # Add predictions to result
        results[model] = {
            'y_true': y,
            'y_pred': pred
        }

    return results

# Feature Importance

In [ ]:
def get_feature_names(preprocessor):
    """Get feature names from a ColumnTransformer."""
    if hasattr(preprocessor, 'get_feature_names_out'):
        return preprocessor.get_feature_names_out()
    else:
        feature_names = []
        for name, transformer, column in preprocessor.transformers_[:-1]:
            if hasattr(transformer, 'get_feature_names'):
                feature_names.extend(transformer.get_feature_names(column))
            else:
                feature_names.extend(column)
        return feature_names

def plot_feature_importance(model_dict, preprocessor, X_train):
    if not hasattr(preprocessor, 'transformers_'):
        preprocessor.fit(X_train)
    
    feature_names = get_feature_names(preprocessor)

    for model_name, model_info in model_dict.items():
        model = model_info['model']

        importances = None

        if 'pipeline' in str(type(model)).lower():
            final_model = model.named_steps['regressor']
            
            # Check for attribute based on the type of model
            if hasattr(final_model, 'coef_'):  # Check for linear models
                importances = np.abs(final_model.coef_.flatten())
            elif hasattr(final_model, 'feature_importances_'):  # Tree-based models
                importances = final_model.feature_importances_
        else:
            # Directly access attributes if not in a pipeline
            if hasattr(model, 'coef_'):
                importances = np.abs(model.coef_.flatten())
            elif hasattr(model, 'feature_importances_'):
                importances = model.feature_importances_

        if importances is None or len(importances) != len(feature_names):
            print(f"No or mismatched feature importance for {model_name}.")
            continue

        importances_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': importances
        }).sort_values(by='Importance', ascending=False)

        plt.figure(figsize=(10, 6))
        plt.title(f'Feature Importance for {model_name}')
        plt.barh(importances_df['Feature'], importances_df['Importance'])
        plt.xlabel('Importance')
        plt.gca().invert_yaxis()
        plt.show()

In [ ]:
def extract_feature_importances(model, preprocessor):
    if 'pipeline' in str(type(model)).lower():
        final_model = model.named_steps['regressor']
        if hasattr(final_model, 'feature_importances_'):
            return final_model.feature_importances_
        elif hasattr(final_model, 'coef_'):
            return np.abs(final_model.coef_.flatten())
    else:
        if hasattr(model, 'feature_importances_'):
            return model.feature_importances_
        elif hasattr(model, 'coef_'):
            return np.abs(model.coef_.flatten())
    return None

In [ ]:
def plot_aggregated_importances(feature_importances, preprocessor):
    feature_names = get_feature_names(preprocessor)

    for model_name, importances_list in feature_importances.items():
        if not importances_list:
            continue

        # Stack importances and calculate mean
        mean_importances = np.mean(np.vstack(importances_list), axis=0)

        # Create DataFrame for plotting
        importances_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': mean_importances
        }).sort_values(by='Importance', ascending=False)

        # Filter to top N features
        top_n = 50  # Show top 20 features
        top_features = importances_df.head(top_n)

        plt.figure(figsize=(12, 10)) 
        plt.title(f'Top {top_n} Feature Importances for {model_name}')
        plt.barh(top_features['Feature'], top_features['Importance'], color='steelblue')
        plt.xlabel('Average Importance')
        plt.xticks(fontsize=10)
        plt.yticks(fontsize=10)
        plt.tight_layout()
        
        plt.savefig(f"{model_name}_feature_importances.png")  
        
        plt.show()

# Run Experiment

In [ ]:

train_inds = pd.read_csv('../input/wt25-dataset/train_inds_tbl.csv')
test_inds = pd.read_csv('../input/wt25-dataset/test_inds_tbl.csv')

# change indices to start from 0 instead of 1
train_inds = train_inds - 1
test_inds = test_inds - 1  
    
error_metrics = {
    'lr': {'rmse': [], 'mae': [], 'r2': [], 'spearman_corr': [], 'pearson_corr': []},
    'lasso': {'rmse': [], 'mae': [], 'r2': [], 'spearman_corr': [], 'pearson_corr': []},
    'rf': {'rmse': [], 'mae': [], 'r2': [], 'spearman_corr': [], 'pearson_corr': []},
    'xgb': {'rmse': [], 'mae': [], 'r2': [], 'spearman_corr': [], 'pearson_corr': []}
}

# Initialize dictionary to store feature importances for each model across all splits
feature_importances = {
    'lr': [],
    'lasso': [],
    'rf': [],
    'xgb': []
}

xgb_all_y = []
xgb_all_pred = []

for i in range(0, 10):
    # Split the dataset
    train_index = train_inds.iloc[:, i]
    test_index = test_inds.iloc[:, i]

    # Extract training and testing sets for X and y
    X_train = X.iloc[train_index]
    y_train = y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test = y.iloc[test_index]
    
    # Transform the data using the pre-fitted preprocessor
    X_train_transformed = preprocessor.transform(X_train)
    X_test_transformed = preprocessor.transform(X_test)

    xgb_study = hyperparam_tuning(X_train_transformed, y_train, X_test_transformed, y_test)
    model_dict = train_models(X_train_transformed, y_train, X_test_transformed, y_test, xgb_study)
    fold_results = evaluate_models(model_dict, X_test_transformed, y_test)
    
    # Collect XGB predictions
    xgb_all_y.extend(fold_results['xgb']['y_true'])
    xgb_all_pred.extend(fold_results['xgb']['y_pred'])

    # extract and store feature importances
    for model_name, model_info in model_dict.items():
        model = model_info['model']
        importances = extract_feature_importances(model, preprocessor) 
        if importances is not None:
            feature_importances[model_name].append(importances)

    # Store the metrics in error_metrics instead of model_dict
    for model_name in model_dict:
        error_metrics[model_name]['rmse'].append(model_dict[model_name]['rmse'])
        error_metrics[model_name]['mae'].append(model_dict[model_name]['mae'])
        error_metrics[model_name]['r2'].append(model_dict[model_name]['r2'])
        error_metrics[model_name]['spearman_corr'].append(model_dict[model_name]['spearman_corr'])
        error_metrics[model_name]['pearson_corr'].append(model_dict[model_name]['pearson_corr'])
        

average_results = {}
for model_name in error_metrics:
    average_results[model_name] = {
        'average_rmse': np.mean(error_metrics[model_name]['rmse']),
        'average_mae': np.mean(error_metrics[model_name]['mae']),
        'average_r2': np.mean(error_metrics[model_name]['r2']),
        'average_spearman_corr': np.mean(error_metrics[model_name]['spearman_corr']),
        'average_pearson_corr': np.mean(error_metrics[model_name]['pearson_corr'])
    }
    print(f"Averages for {model_name}:")
    print(f"  RMSE: {average_results[model_name]['average_rmse']}")
    print(f"  MAE: {average_results[model_name]['average_mae']}")
    print(f"  R2: {average_results[model_name]['average_r2']}")
    print(f"  spearman_corr: {average_results[model_name]['average_spearman_corr']}")
    print(f"  pearson_corr: {average_results[model_name]['average_pearson_corr']}")

plot_aggregated_importances(feature_importances, preprocessor)

In [ ]:
# XGB Dot-Plot

# Fit linear regression
slope, intercept = np.polyfit(xgb_all_y, xgb_all_pred, 1)
x_vals = np.array(xgb_all_y)
line = slope * x_vals + intercept

plt.figure(figsize=(8, 6))
plt.scatter(xgb_all_y, xgb_all_pred, alpha=0.5, edgecolors='k')
plt.plot(xgb_all_y, line, color='red', linewidth=2)
plt.xlabel("True Readthrough (RE)")
plt.ylabel("Predicted Readthrough")
plt.title("XGBoost True vs. Predicted")
plt.grid(True)
plt.tight_layout()
plt.show()

# Dot Plots of RE and specific features

In [ ]:
# Get tAI_70 values
x = X['tAI_70']

# Calculate Spearman correlation
rho, pval = spearmanr(x, y)

# Fit linear regression
slope, intercept = np.polyfit(x, y, 1)
line = slope * x + intercept

# Plot
plt.figure(figsize=(8, 6))
plt.scatter(x, y, alpha=0.6)
plt.plot(x, line, color='red', linewidth=2)

# Annotate with Spearman correlation info (top-right corner)
plt.text(
    0.95, 0.95, f"Spearman’s ρ = {rho:.2f}\np-value = {pval:.2e}",
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment='top',
    horizontalalignment='right',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.5)
)

plt.xlabel('tAI_70')
plt.ylabel('RE')
plt.title('tAI_70 vs RE')
plt.grid(True)
plt.show()


# tAI_70 for top 20% RC
top20RC = pd.read_csv('../input/top20rc/top20RC.csv')
tAI70 = top20RC['tAI_70'].copy()
RE = top20RC['RE'].copy()

# Calculate Spearman correlation
rho, pval = spearmanr(tAI70, RE)

# Fit linear regression
slope, intercept = np.polyfit(tAI70, RE, 1)
line = slope * tAI70 + intercept

# Plot
plt.figure(figsize=(8, 6))
plt.scatter(tAI70, RE, alpha=0.6)
plt.plot(tAI70, line, color='red', linewidth=2)

# Annotate with Spearman correlation info (top-right corner)
plt.text(
    0.95, 0.95, f"Spearman’s ρ = {rho:.2f}\np-value = {pval:.2e}",
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment='top',
    horizontalalignment='right',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.5)
)

plt.xlabel('tAI_70')
plt.ylabel('RE')
plt.title('tAI_70 vs RE for top 20% RC genes')
plt.grid(True)
plt.show()


# Get tAI_50 values
x = X['tAI_50']

# Calculate Spearman correlation
rho, pval = spearmanr(x, y)

# Fit linear regression
slope, intercept = np.polyfit(x, y, 1)
line = slope * x + intercept

# Plot
plt.figure(figsize=(8, 6))
plt.scatter(x, y, alpha=0.6)
plt.plot(x, line, color='red', linewidth=2)

# Annotate with Spearman correlation info (top-right corner)
plt.text(
    0.95, 0.95, f"Spearman’s ρ = {rho:.2f}\np-value = {pval:.2e}",
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment='top',
    horizontalalignment='right',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.5)
)

plt.xlabel('tAI_50')
plt.ylabel('RE')
plt.title('tAI_50 vs RE')
plt.grid(True)
plt.show()


# Get tAI_30 values
x = X['tAI_30']

# Calculate Spearman correlation
rho, pval = spearmanr(x, y)

# Fit linear regression
slope, intercept = np.polyfit(x, y, 1)
line = slope * x + intercept

# Plot
plt.figure(figsize=(8, 6))
plt.scatter(x, y, alpha=0.6)
plt.plot(x, line, color='red', linewidth=2)

# Annotate with Spearman correlation info (top-right corner)
plt.text(
    0.95, 0.95, f"Spearman’s ρ = {rho:.2f}\np-value = {pval:.2e}",
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment='top',
    horizontalalignment='right',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.5)
)

plt.xlabel('tAI_30')
plt.ylabel('RE')
plt.title('tAI_30 vs RE')
plt.grid(True)
plt.show()


# Get CAI_70 values
x = X['CAI_70']

# Calculate Spearman correlation
rho, pval = spearmanr(x, y)

# Fit linear regression
slope, intercept = np.polyfit(x, y, 1)
line = slope * x + intercept

# Plot
plt.figure(figsize=(8, 6))
plt.scatter(x, y, alpha=0.6)
plt.plot(x, line, color='red', linewidth=2)

# Annotate with Spearman correlation info (top-right corner)
plt.text(
    0.95, 0.95, f"Spearman’s ρ = {rho:.2f}\np-value = {pval:.2e}",
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment='top',
    horizontalalignment='right',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.5)
)

plt.xlabel('CAI_70')
plt.ylabel('RE')
plt.title('CAI_70 vs RE')
plt.grid(True)
plt.show()


# Get CAI_50 values
x = X['CAI_50']

# Calculate Spearman correlation
rho, pval = spearmanr(x, y)

# Fit linear regression
slope, intercept = np.polyfit(x, y, 1)
line = slope * x + intercept

# Plot
plt.figure(figsize=(8, 6))
plt.scatter(x, y, alpha=0.6)
plt.plot(x, line, color='red', linewidth=2)

# Annotate with Spearman correlation info (top-right corner)
plt.text(
    0.95, 0.95, f"Spearman’s ρ = {rho:.2f}\np-value = {pval:.2e}",
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment='top',
    horizontalalignment='right',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.5)
)

plt.xlabel('CAI_50')
plt.ylabel('RE')
plt.title('CAI_50 vs RE')
plt.grid(True)
plt.show()


# Get CAI_30 values
x = X['CAI_30']

# Calculate Spearman correlation
rho, pval = spearmanr(x, y)

# Fit linear regression
slope, intercept = np.polyfit(x, y, 1)
line = slope * x + intercept

# Plot
plt.figure(figsize=(8, 6))
plt.scatter(x, y, alpha=0.6)
plt.plot(x, line, color='red', linewidth=2)

# Annotate with Spearman correlation info (top-right corner)
plt.text(
    0.95, 0.95, f"Spearman’s ρ = {rho:.2f}\np-value = {pval:.2e}",
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment='top',
    horizontalalignment='right',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.5)
)

plt.xlabel('CAI_30')
plt.ylabel('RE')
plt.title('CAI_30 vs RE')
plt.grid(True)
plt.show()


# Get similar_to_stop_cnt values
x = X['similar_to_stop_cnt']

# Calculate Spearman correlation
rho, pval = spearmanr(x, y)

# Fit linear regression
slope, intercept = np.polyfit(x, y, 1)
line = slope * x + intercept

# Plot
plt.figure(figsize=(8, 6))
plt.scatter(x, y, alpha=0.6)
plt.plot(x, line, color='red', linewidth=2)

# Annotate with Spearman correlation info (top-right corner)
plt.text(
    0.95, 0.95, f"Spearman’s ρ = {rho:.2f}\np-value = {pval:.2e}",
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment='top',
    horizontalalignment='right',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.5)
)

plt.xlabel('similar_to_stop_cnt')
plt.ylabel('RE')
plt.title('similar_to_stop_cnt vs RE')
plt.grid(True)
plt.show()

# SHAP

In [ ]:
# Sample and preprocess input
X_sample = X.sample(n=200, random_state=42)
X_sample_transformed = preprocessor.transform(X_sample)
X_sample_dense = X_sample_transformed.toarray()

# Get XGBoost model (not in pipeline)
xgb_model = model_dict['xgb']['model']

# Use same sample and transformed input
explainer_xgb = shap.Explainer(xgb_model, X_sample_dense)
shap_values_xgb = explainer_xgb(X_sample_dense)

# SHAP summary plot
shap.summary_plot(shap_values_xgb.values, features=X_sample_dense, feature_names=feature_names, show=False)
plt.title("SHAP Summary Plot – XGB", fontsize=14)
plt.tight_layout()
plt.show()